# 05 — Extracting Text from Images & Scanned Docs (OCR + Vision LLMs)

## What you'll learn

Some source material only exists as pixels — screenshots, scanned forms, photographed
receipts and whiteboards. This notebook pulls text out of 5 synthetic test images spanning
that difficulty spectrum, using two fundamentally different approaches: classic OCR
(Tesseract, pattern-matches glyphs) and a local vision LLM (gemma3:4b via Ollama, reasons about
the image). You'll see where each wins and what it costs.


## Architecture

![Architecture diagram](https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgICBHW2dlbmVyYXRlX2RhdGEucHldIC0tPiBJTUdTWzUgdGVzdCBpbWFnZXM6PGJyLz5zY3JlZW5zaG90IC8gc2NhbiAvIHJlY2VpcHQgLzxici8-d2hpdGVib2FyZCAvIGxvdy1yZXNdCiAgICBJTUdTIC0tPiBPQ1IxW1Rlc3NlcmFjdDogcmF3XQogICAgSU1HUyAtLT4gT0NSMltUZXNzZXJhY3Q6IGdyYXlzY2FsZSArIHRocmVzaG9sZF0KICAgIElNR1MgLS0-IFZJU1siZ2VtbWEzOjRiIHZpc2lvbiAoT2xsYW1hKSJdCiAgICBPQ1IxICYgT0NSMiAmIFZJUyAtLT4gQ01QW0NvbXBhcmUgdGV4dCArIGxhdGVuY3ldCiAgICBDTVAgLS0-IEpbZXh0cmFjdGVkX2NodW5rcy5qc29uPGJyLz50ZXh0ICsgaW1hZ2VfcGF0aCBtZXRhZGF0YV0?theme=dark)


## Setup

Runs fully local via [Ollama](https://ollama.com) — no API key needed. Start the Ollama server and pull the model(s) first:

```bash
ollama pull gemma3:4b
```

This episode also requires
`tesseract` to be installed locally (it's included via `conda-forge` in `environment.yml`).


In [15]:
from pathlib import Path
from dotenv import load_dotenv

# override=True so re-running this cell picks up edits to .env without restarting the kernel
load_dotenv(override=True)

DATA_DIR = Path("data")
OUT_PATH = DATA_DIR / "extracted_chunks.json"

## Generate sample data

Synthesizes 5 test images covering the OCR difficulty spectrum: a clean screenshot, a
"scanned" page (rotated + noisy), a receipt, a whiteboard photo (angled, marker-like
strokes), and a low-res scan (blurred + downsampled). These are gitignored — rerun this
cell any time you need them back.


In [16]:
from pathlib import Path
from PIL import Image
from PIL import ImageDraw
from PIL import ImageFilter
from PIL import ImageFont
import random

random.seed(42)  # deterministic jitter so the whiteboard demo looks the same on every run


def _font(size: int) -> ImageFont.ImageFont:
    # try real system fonts first — PIL's bitmap default font looks nothing like real text
    # and would make OCR/vision comparisons meaningless
    for candidate in ["/System/Library/Fonts/Supplemental/Arial.ttf",
                      "/System/Library/Fonts/Helvetica.ttc"]:
        if Path(candidate).exists():
            return ImageFont.truetype(candidate, size)
    return ImageFont.load_default()


def make_clean_screenshot():
    img = Image.new("RGB", (700, 300), "white")
    draw = ImageDraw.Draw(img)
    lines = [
        "def calculate_total(items):",
        '    """Sum item prices after a 7% tax."""',
        "    subtotal = sum(item.price for item in items)",
        "    return round(subtotal * 1.07, 2)",
    ]
    for i, line in enumerate(lines):
        draw.text((20, 20 + i * 30), line, fill="black", font=_font(18))
    img.save(DATA_DIR / "clean_screenshot.png")
    print("wrote clean_screenshot.png")


def make_scanned_page():
    img = Image.new("L", (700, 500), 245)
    draw = ImageDraw.Draw(img)
    lines = [
        "Warehouse Safety Bulletin",
        "",
        "All forklift operators must complete recertification",
        "training by the end of this quarter. Sessions are held",
        "every Tuesday at 9 AM in the east training room.",
    ]
    for i, line in enumerate(lines):
        draw.text((40, 40 + i * 35), line, fill=20, font=_font(22))
    # simulate scan noise + slight skew
    img = img.rotate(1.5, expand=True, fillcolor=245)
    noise = Image.effect_noise(img.size, 18)
    # blend at a low alpha so the noise is subtle grain, not enough to obscure the text
    img = Image.blend(img, noise, alpha=0.05)
    img.convert("RGB").save(DATA_DIR / "scanned_page.png")
    print("wrote scanned_page.png")


def make_receipt():
    img = Image.new("L", (300, 500), 250)
    draw = ImageDraw.Draw(img)
    lines = [
        "CORNER MARKET",
        "12 Maple St",
        "------------------------",
        "Bread          3.49",
        "Milk 1gal      4.19",
        "Eggs dozen     3.99",
        "Coffee beans   9.50",
        "------------------------",
        "TOTAL         21.17",
    ]
    for i, line in enumerate(lines):
        draw.text((15, 20 + i * 28), line, fill=10, font=_font(16))
    # thermal-receipt paper is rarely perfectly sharp — a light blur mimics that
    img = img.filter(ImageFilter.GaussianBlur(0.4))
    img.convert("RGB").save(DATA_DIR / "receipt.png")
    print("wrote receipt.png")


def make_whiteboard():
    img = Image.new("RGB", (600, 400), (250, 250, 245))
    draw = ImageDraw.Draw(img)
    lines = ["Sprint Goals:", "- ship OCR demo", "- fix flaky test", "- review PR #42"]
    for i, line in enumerate(lines):
        # per-line horizontal jitter mimics the unevenness of real handwriting/marker strokes
        x_jitter = random.randint(-3, 3)
        draw.text((60 + x_jitter, 40 + i * 60), line, fill=(30, 60, 140), font=_font(28))
    img = img.rotate(-2, expand=True, fillcolor=(250, 250, 245))
    img = img.filter(ImageFilter.GaussianBlur(0.6))
    img.save(DATA_DIR / "whiteboard.png")
    print("wrote whiteboard.png")


def make_low_res_scan():
    img = Image.new("L", (700, 300), 240)
    draw = ImageDraw.Draw(img)
    draw.text((30, 30), "Return Authorization Form", fill=15, font=_font(24))
    draw.text((30, 80), "Order #: 88213", fill=15, font=_font(20))
    draw.text((30, 120), "Reason: item damaged in transit", fill=15, font=_font(20))
    # downsample then upsample to simulate a low-res scan, then blur
    small = img.resize((175, 75))
    low_res = small.resize((700, 300))
    low_res = low_res.filter(ImageFilter.GaussianBlur(1.2))
    low_res.convert("RGB").save(DATA_DIR / "low_res_scan.png")
    print("wrote low_res_scan.png")


make_clean_screenshot()
make_scanned_page()
make_receipt()
make_whiteboard()
make_low_res_scan()
print("\nDone. These 5 images are gitignored — rerun this cell any time.")

wrote clean_screenshot.png
wrote scanned_page.png
wrote receipt.png
wrote whiteboard.png
wrote low_res_scan.png

Done. These 5 images are gitignored — rerun this cell any time.


## `ocr_extract` — classic OCR with Tesseract

Cheap (free, runs locally) and fast, but sensitive to noise, skew, and low resolution.
Preprocessing (grayscale + thresholding) often recovers accuracy that raw Tesseract loses
on scanned or low-quality images.


In [17]:
from pathlib import Path
import pytesseract
from PIL import Image
from PIL import ImageOps


def ocr_raw(image_path: Path) -> str:
    """Run Tesseract directly on the image, no preprocessing."""
    return pytesseract.image_to_string(Image.open(image_path)).strip()


def ocr_preprocessed(image_path: Path) -> str:
    """Grayscale + binarize (simple fixed threshold) before OCR — this is
    the single highest-leverage preprocessing step for scanned documents."""
    img = Image.open(image_path).convert("L")  # grayscale
    # stretch the contrast range before thresholding, so faint scans don't
    # collapse entirely to one color once we binarize
    img = ImageOps.autocontrast(img)
    threshold = 150
    # hard black/white cutoff — removes anti-aliasing/noise that confuses Tesseract's
    # character segmentation, at the cost of being sensitive to the threshold value
    img = img.point([255 if value > threshold else 0 for value in range(256)])
    return pytesseract.image_to_string(img).strip()

In [18]:
for image_path in sorted(DATA_DIR.glob("*.png")):
    print(f"\n--- {image_path.name} ---")
    print("raw:          ", ocr_raw(image_path)[:80].replace("\n", " "))
    print("preprocessed: ", ocr_preprocessed(image_path)[:80].replace("\n", " "))


--- clean_screenshot.png ---
raw:           def calculate_total(items): “"Sum item prices after a 7% tax." subtotal = sum(it
preprocessed:  def calculate_total(items): “Sum item prices after a 7% tax.” subtotal = sum(ite

--- low_res_scan.png ---
raw:           Return Athorzation form Order © S821) Meancr tem temeget ¢ reret
preprocessed:  Retr Ashoreahon form Onan © 0521) Reese Ger Ge~aged © reret

--- receipt.png ---
raw:           CORNER MARKET 12 Maple St  Bread 3.49 Milk 1gal_ 4.19  Eggs dozen 3.99 Coffee be
preprocessed:  CORNER MARKET 12 Maple St  Bread 3.49 Milk tgal 4.19  Eggs dozen 3.99 Coffee bea

--- scanned_page.png ---
raw:           Warehouse Safety Bulletin  All forklift operators must complete recertification 
preprocessed:  Warehouse Safety Bulletin  All forklift operators must complete recertification 

--- whiteboard.png ---
raw:           Sprint Goals: - ship OCR demo - fix flaky test  - review PR #42
preprocessed:  Sprint Goals: - ship OCR demo - fix flaky test  

## `vision_llm_extract` — transcribe images with a vision LLM instead of classic OCR

Needs a GPU/CPU-heavy local model and is much slower than Tesseract, but handles skew, low resolution, and
messy handwriting far better because it's reasoning about the image, not just
pattern-matching glyphs. `gemma3:4b` is a multimodal model that accepts image input.


In [19]:
import base64
from pathlib import Path
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama

TRANSCRIBE_PROMPT = (
    "Transcribe all text visible in this image exactly as written. "
    "If it looks like handwriting or a whiteboard, still transcribe your best "
    "reading. Output only the transcribed text, nothing else."
)


def vision_transcribe(image_path: Path) -> str:
    # vision models take images as inline base64 data URIs, not file paths
    b64 = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    llm = ChatOllama(model="gemma3:4b", temperature=0, reasoning=False)
    # multimodal message: a text part (the instruction) plus an image part in one HumanMessage
    message = HumanMessage(
        content=[
            {"type": "text", "text": TRANSCRIBE_PROMPT},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
        ]
    )
    return llm.invoke([message]).content.strip()

In [20]:
for image_path in sorted(DATA_DIR.glob("*.png")):
    print(f"\n--- {image_path.name} ---")
    print(vision_transcribe(image_path))


--- clean_screenshot.png ---
def calculate_total(items):
  """Sum item prices after a 7% tax."""
  subtotal = sum(item.price for item in items)
  return round(subtotal * 1.07, 2)

--- low_res_scan.png ---
Return Authorization Form
Order # 88213
Reason: Item damaged in transit

--- receipt.png ---
CORNER MARKET
12 Maple St

Bread 3.49
Milk 1gal 4.19
Eggs dozen 3.99
Coffee beans 9.50

TOTAL 21.17

--- scanned_page.png ---
Warehouse Safety Bulletin

All forklift operators must complete recertification training by the end of this quarter. Sessions are held every Tuesday at 9 AM in the east training room.

--- whiteboard.png ---
Sprint Goals:

- ship OCR demo
- fix flaky test
- review PR #42


## `compare` — Tesseract (raw + preprocessed) vs. vision LLM on all 5 images

Print outputs side by side, then store the best result per image as a chunk with
image-path metadata (mirrors what an ingestion pipeline would write into a vector store).


In [21]:
import time
from pathlib import Path


def compare_one(image_path: Path) -> dict:
    # wall-clock timing for each method so we can weigh accuracy against latency/cost,
    # not just compare the transcribed text
    t0 = time.time()
    raw = ocr_raw(image_path)
    t_ocr_raw = time.time() - t0

    t0 = time.time()
    preprocessed = ocr_preprocessed(image_path)
    t_ocr_pre = time.time() - t0

    t0 = time.time()
    vision = vision_transcribe(image_path)
    t_vision = time.time() - t0

    return {
        "image": image_path.name,
        "ocr_raw": {"text": raw, "seconds": round(t_ocr_raw, 2)},
        "ocr_preprocessed": {"text": preprocessed, "seconds": round(t_ocr_pre, 2)},
        "vision_llm": {"text": vision, "seconds": round(t_vision, 2)},
    }

In [22]:
import json

images = sorted(DATA_DIR.glob("*.png"))

results = []
for image_path in images:
    print(f"\n{'=' * 70}\n{image_path.name}\n{'=' * 70}")
    result = compare_one(image_path)
    results.append(result)
    for method in ["ocr_raw", "ocr_preprocessed", "vision_llm"]:
        r = result[method]
        snippet = r["text"][:100].replace("\n", " ")
        print(f"  {method:16s} ({r['seconds']:.2f}s): {snippet}")

# Store the vision-LLM transcription as the "best result" chunk — for
# noisy/handwritten sources it consistently beats raw OCR, at higher cost
chunks = [
    {
        "text": r["vision_llm"]["text"],
        "metadata": {
            "image_path": r["image"],
            "extraction_method": "vision_llm",
        },
    }
    for r in results
]
OUT_PATH.write_text(json.dumps(chunks, indent=2))
print(f"\nWrote {len(chunks)} chunks -> {OUT_PATH}")


clean_screenshot.png
  ocr_raw          (0.21s): def calculate_total(items): “"Sum item prices after a 7% tax." subtotal = sum(item.price for item in
  ocr_preprocessed (0.12s): def calculate_total(items): “Sum item prices after a 7% tax.” subtotal = sum(item.price for item in 
  vision_llm       (3.62s): def calculate_total(items):   """Sum item prices after a 7% tax."""   subtotal = sum(item.price for 

low_res_scan.png
  ocr_raw          (0.19s): Return Athorzation form Order © S821) Meancr tem temeget ¢ reret
  ocr_preprocessed (0.15s): Retr Ashoreahon form Onan © 0521) Reese Ger Ge~aged © reret
  vision_llm       (2.01s): Return Authorization Form Order # 88213 Reason: Item damaged in transit

receipt.png
  ocr_raw          (0.51s): CORNER MARKET 12 Maple St  Bread 3.49 Milk 1gal_ 4.19  Eggs dozen 3.99 Coffee beans 9.50  TOTAL 2117
  ocr_preprocessed (0.15s): CORNER MARKET 12 Maple St  Bread 3.49 Milk tgal 4.19  Eggs dozen 3.99 Coffee beans 9.50  TOTAL 21.17
  vision_llm       (3

## Run it end-to-end

Running the cells above top to bottom already reproduces what `main.py` used to do: generate
the 5 sample images, then run the full OCR-vs-vision-LLM comparison and write
`extracted_chunks.json`. The cell below just confirms the output file landed.


In [23]:
print("Comparison complete for", len(results), "images.")
print("Chunks file exists:", OUT_PATH.exists(), "->", OUT_PATH)

Comparison complete for 5 images.
Chunks file exists: True -> data/extracted_chunks.json


## Key concepts

1. **Classic OCR vs. vision LLMs** — Tesseract is free, fast, and runs locally, but is
   brittle to noise, skew, and handwriting; a vision LLM is slower and needs far more compute but reasons
   about the whole image, not just isolated glyphs.
2. **Preprocessing for OCR** — grayscale conversion and thresholding (binarizing pixels to
   pure black/white) is the single highest-leverage fix for scanned-document accuracy,
   demonstrated directly in `ocr_preprocessed`.
3. **Handwriting & low-quality scans** — this is where the OCR/vision-LLM accuracy gap
   widens the most; the whiteboard and low-res images make that visible.
4. **Transcribe vs. caption** — for a receipt you want exact transcription; for a photo of
   a whiteboard diagram you might want a _description_ instead — the prompt controls which.


## Try this yourself

1. Lower the threshold value in `ocr_preprocessed()` (try 100 instead of 150) and see how
   it changes Tesseract's accuracy on `scanned_page.png`.
2. Compare the per-image latency printed above — how much slower is the vision LLM than
   Tesseract, and does that gap matter more or less for a one-time batch ingest vs. a
   real-time upload flow?
3. Change `TRANSCRIBE_PROMPT` to ask for a _description_ instead of a transcription on
   `whiteboard.png`, and compare usefulness for search.


## Common pitfalls

- **Assuming OCR accuracy is binary** — it degrades gradually with noise/skew/resolution;
  always spot-check a sample before trusting an OCR pipeline at scale.
- **Skipping preprocessing** — raw Tesseract on a noisy scan is often dramatically worse
  than the same image after a two-line grayscale+threshold pass.
- **Using vision LLMs for everything by default** — on a large batch of clean documents,
  spending seconds of local GPU/CPU time per image on what Tesseract already handles in
  milliseconds is wasted compute; reserve vision LLMs for the images that actually need them.
